# MiniMax H3 — ComfyUI + Cloudflare (Server-First)

Main no-MCP notebook.

Order:
1. Mount Drive
2. Check GPU
3. Clone/update repo
4. Install ComfyUI + H3 custom nodes
5. Load Cloudflare secret
6. Start ComfyUI + Cloudflare immediately
7. Verify server/tunnel
8. Download H3 models
9. Restart ComfyUI once so all new models appear

No workflow-restore section. No LTX section.


## 0. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PERSIST_MODELS_TO_DRIVE = False
PERSIST_OUTPUT_TO_DRIVE = True
DRIVE_ROOT = '/content/drive/MyDrive/MiniMax_H3_ComfyUI'


## 1. Check GPU

In [ ]:
import subprocess, os

def sh(cmd):
    return subprocess.check_output(cmd, shell=True, text=True).strip()

gpu_name = sh('nvidia-smi --query-gpu=name --format=csv,noheader | head -n1')
vram_mb = int(sh('nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | head -n1'))
name = gpu_name.lower()
if 'rtx pro 6000' in name or 'blackwell' in name:
    H3_RESERVE_VRAM_GB = '6'
elif 'a100' in name:
    H3_RESERVE_VRAM_GB = '4'
elif 'l4' in name:
    H3_RESERVE_VRAM_GB = '2'
else:
    H3_RESERVE_VRAM_GB = '1'
print(f'GPU: {gpu_name} ({vram_mb/1024:.1f} GB)')
print('Reserved VRAM:', H3_RESERVE_VRAM_GB, 'GB')


## 2. Clone/update H3 branch

In [ ]:
%cd /content
!rm -rf /content/All-testing /content/minimax_h3_comfy
!git clone --depth 1 --branch minimax-h3-colab https://github.com/Logan17de/All-testing.git /content/All-testing
!cp -r /content/All-testing/video/minimax_h3_comfy /content/minimax_h3_comfy
%cd /content/minimax_h3_comfy


## 3. Install/update ComfyUI + H3 custom nodes

In [ ]:
import os, subprocess, sys

os.environ['COMFY_ROOT'] = '/content/ComfyUI'
os.environ['H3_DRIVE_ROOT'] = DRIVE_ROOT
os.environ['H3_PERSIST_MODELS'] = '1' if PERSIST_MODELS_TO_DRIVE else '0'
os.environ['H3_PERSIST_OUTPUT'] = '1' if PERSIST_OUTPUT_TO_DRIVE else '0'
os.environ['H3_VRAM_MODE'] = 'auto'
os.environ['H3_RESERVE_VRAM_GB'] = H3_RESERVE_VRAM_GB
os.environ['H3_PREVIEW_METHOD'] = 'none'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!bash install_comfy_h3.sh

CUSTOM='/content/ComfyUI/custom_nodes'
def clone_or_pull(url, folder):
    path=f'{CUSTOM}/{folder}'
    if os.path.isdir(path+'/.git'):
        subprocess.run(['git','-C',path,'pull','--ff-only'], check=False)
    else:
        subprocess.run(['git','clone','--depth','1',url,path], check=True)
    req=os.path.join(path,'requirements.txt')
    if os.path.exists(req):
        subprocess.run([sys.executable,'-m','pip','install','-r',req], check=False)

clone_or_pull('https://github.com/AIMixer/ComfyUI_MiniMaxH3_Director.git','ComfyUI_MiniMaxH3_Director')
clone_or_pull('https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git','ComfyUI-VideoHelperSuite')
clone_or_pull('https://github.com/kijai/ComfyUI-KJNodes.git','ComfyUI-KJNodes')
clone_or_pull('https://github.com/pixaroma/ComfyUI-Pixaroma.git','ComfyUI-Pixaroma')
subprocess.run([sys.executable,'-m','pip','install','-U','huggingface_hub'], check=True)
print('✅ ComfyUI + H3 custom nodes ready.')


## 4. Load Cloudflare secret + install cloudflared

Cloudflare published application must be `comfy.zetbros.com` → `http://127.0.0.1:8188`.

In [ ]:
import os, platform, re, subprocess
from google.colab import userdata

raw = userdata.get('CF_TUNNEL_TOKEN')
if not raw:
    raise RuntimeError('CF_TUNNEL_TOKEN is missing from Colab Secrets.')
m = re.search(r'(eyJ[A-Za-z0-9._=-]+)', raw)
cf_token = m.group(1) if m else raw.strip()
if not cf_token.startswith('eyJ'):
    raise RuntimeError('CF_TUNNEL_TOKEN does not look like a Cloudflare tunnel token.')
os.environ['CLOUDFLARE_TUNNEL_TOKEN'] = cf_token
os.environ['CLOUDFLARE_COMFY_URL'] = 'https://comfy.zetbros.com'
arch = platform.machine().lower()
cf_arch = 'amd64' if arch in ('x86_64','amd64') else 'arm64' if arch in ('aarch64','arm64') else None
if not cf_arch:
    raise RuntimeError(f'Unsupported architecture: {arch}')
if subprocess.run(['bash','-lc','command -v cloudflared >/dev/null 2>&1']).returncode != 0:
    url = f'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{cf_arch}'
    subprocess.run(['curl','-fL','--retry','3','--retry-delay','2',url,'-o','/usr/local/bin/cloudflared'], check=True)
    subprocess.run(['chmod','0755','/usr/local/bin/cloudflared'], check=True)
print('✅ Cloudflare secret loaded.')
print(subprocess.check_output(['cloudflared','--version'], text=True).strip())


## 5. Start ComfyUI + Cloudflare now

This happens before the large model downloads. Open `https://comfy.zetbros.com` after this cell succeeds.

In [ ]:
%cd /content/minimax_h3_comfy
!bash launch_comfy_cloudflare.sh


## 6. Verify live server/tunnel

In [ ]:
import subprocess
local = subprocess.run(['curl','-fsS','--connect-timeout','2','--max-time','5','http://127.0.0.1:8188/system_stats'], capture_output=True)
print('Local ComfyUI:', 'HTTP OK' if local.returncode == 0 else 'FAILED')
subprocess.run(['bash','-lc',"ss -ltnp | grep ':8188' || true"])
subprocess.run(['bash','-lc',"pgrep -af 'cloudflared.*tunnel.*run' | sed -E 's/(--token )[A-Za-z0-9._=-]+/\1[REDACTED]/g' || true"])
print('\n🌐 https://comfy.zetbros.com')
print('The UI can open now. H3 loaders will populate after Section 7 + the final restart.')


## 7. Download H3 models while the server stays online

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
import shutil

MODEL_ROOT = Path(f'{DRIVE_ROOT}/models' if PERSIST_MODELS_TO_DRIVE else '/content/ComfyUI/models')
for folder in ['diffusion_models','text_encoders','vae','loras','latent_upscale_models','upscale_models']:
    (MODEL_ROOT/folder).mkdir(parents=True, exist_ok=True)

if PERSIST_MODELS_TO_DRIVE:
    for folder in ['diffusion_models','text_encoders','vae','loras','latent_upscale_models','upscale_models']:
        local = Path('/content/ComfyUI/models')/folder
        drive_dir = MODEL_ROOT/folder
        drive_dir.mkdir(parents=True, exist_ok=True)
        if local.is_symlink(): local.unlink()
        elif local.exists():
            shutil.rmtree(local) if local.is_dir() else local.unlink()
        local.symlink_to(drive_dir, target_is_directory=True)

downloads = [
 ('Comfy-Org/MiniMax-H3','diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors',MODEL_ROOT,MODEL_ROOT/'diffusion_models'/'minimax_h3_fl2va_pruned_int8_convrot.safetensors'),
 ('Comfy-Org/MiniMax-H3','diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors',MODEL_ROOT,MODEL_ROOT/'diffusion_models'/'minimax_h3_ref2va_pruned_int8_convrot.safetensors'),
 ('Comfy-Org/MiniMax-H3','text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',MODEL_ROOT,MODEL_ROOT/'text_encoders'/'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'),
 ('Comfy-Org/MiniMax-H3','vae/minimax_h3_video_vae_fp16.safetensors',MODEL_ROOT,MODEL_ROOT/'vae'/'minimax_h3_video_vae_fp16.safetensors'),
 ('Comfy-Org/MiniMax-H3','vae/minimax_h3_audio_vae_fp32.safetensors',MODEL_ROOT,MODEL_ROOT/'vae'/'minimax_h3_audio_vae_fp32.safetensors'),
 ('lightx2v/Minimax-h3-Turbo','minimax_h3_ref2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors',MODEL_ROOT/'loras',MODEL_ROOT/'loras'/'minimax_h3_ref2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors'),
 ('lightx2v/Minimax-h3-Turbo','minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors',MODEL_ROOT/'loras',MODEL_ROOT/'loras'/'minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors'),
 ('LBH-123-AI/Minimax_h3_latent_Upscaler','minimax_h3_latent_upscaler_3d_fp16.safetensors',MODEL_ROOT/'latent_upscale_models',MODEL_ROOT/'latent_upscale_models'/'minimax_h3_latent_upscaler_3d_fp16.safetensors')
]

for repo_id, filename, local_dir, target in downloads:
    if target.exists() and target.stat().st_size > 1024*1024:
        print('SKIP', target.name)
        continue
    print('DOWNLOAD', filename)
    hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(local_dir))
    if not target.exists():
        raise FileNotFoundError(target)
    print('READY', target.name)

print('✅ H3 model stack downloaded.')


## 8. Final one-time restart

Refreshes ComfyUI's model lists. The public hostname remains `https://comfy.zetbros.com`.

In [ ]:
import os, subprocess, time
subprocess.run(['bash','-lc',"pkill -f 'python.*main.py.*--port 8188' || true"])
time.sleep(2)
os.chdir('/content/minimax_h3_comfy')
subprocess.run(['bash','launch_comfy_cloudflare.sh'], check=True)
print('✅ FINAL READY')
print('🌐 https://comfy.zetbros.com')
print('Load your workflow manually in ComfyUI.')


## Diagnostics

In [ ]:
import subprocess
print('=== LOCAL COMFY ===')
subprocess.run(['bash','-lc',"curl -sS -o /dev/null -w 'HTTP %{http_code}\n' http://127.0.0.1:8188/system_stats || true"])
subprocess.run(['bash','-lc',"ss -ltnp | grep ':8188' || true"])
print('\n=== CLOUDFLARED ===')
subprocess.run(['bash','-lc',"pgrep -af 'cloudflared.*tunnel.*run' | sed -E 's/(--token )[A-Za-z0-9._=-]+/\1[REDACTED]/g' || true"])
subprocess.run(['bash','-lc','tail -n 60 /content/h3_comfy_logs/cloudflared.log || true'])
print('\n=== COMFY LOG ===')
subprocess.run(['bash','-lc','tail -n 80 /content/h3_comfy_logs/comfyui.log || true'])
